### Analysis of retrograde tracing experiment
-> Retrobeads were injected into the MD and the RE and we imaged the mPFC for overlap


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure, img_as_float
from skimage.color import rgb2hsv, hsv2rgb
import os

import sys
from datetime import datetime
from pathlib import Path

# Add the parent directory to the system path
current_dir = Path().resolve()
sys.path.append(str(current_dir.parent.parent))

import fijifuns as fj
import tifffile as tf

from matplotlib.patches import Circle

Beads 12


In [2]:
# crop the image using the GUI
cropped, file_path = fj.crop_tiff_gui()

In [7]:
cropped.shape


AttributeError: 'NoneType' object has no attribute 'shape'

Beads 9

In [ ]:
# crop the image using the GUI
cropped, file_path = fj.crop_tiff_gui()

# check to save if a cropped image is already saved
file_save = file_path.replace(".tif", "_cropped.tif")
if os.path.exists(file_save):
    # add a digit
    i = 1
    while os.path.exists(file_save.replace("_cropped.tif", f"_cropped_{i}.tif")):
        i += 1
    file_path = file_path.replace(".tif", f"_cropped_{i}.tif")
else:
    file_path = file_path.replace(".tif", "_cropped.tif")
tf.imsave(file_path, cropped)

# adjust
adjusted = fj.rescale_image(cropped)

# gen a save path for figures
fig_save_path = os.path.split(file_path)[0]

# save the cropped image
fig1, axs1 = plt.subplots(figsize=(8, 8))
axs1.imshow(adjusted)
axs1.axis('off')
fig1.savefig(os.path.join(fig_save_path,'CroppedOverlay_HSVstretch.eps'),
            format='eps',
            dpi=1000, # controls raster‐embedded image resolution
            bbox_inches='tight') 

# plot OG
og_img = tf.imread(file_path.replace("_cropped.tif", ".tif"))
og_img = fj.rescale_image(og_img)
#og_img_crop = crop_img_with_gui(og_img)
#fig2, axs2 = plt.subplots(figsize=(8, 8))
#axs2.imshow(og_img)
#axs2.axis('off')
#fig2.savefig(os.path.join(fig_save_path,'OGimg_HSVstretch.eps'),
#            format='eps',
#            dpi=1000, # controls raster‐embedded image resolution
#            bbox_inches='tight')     


# Separate channels
cropped_red = cropped[:,:,0]
cropped_green = cropped[:,:,1]
cropped_blue = cropped[:,:,2]
new_path_red = os.path.splitext(file_path)[0] + "_cropped_red.tif"
new_path_green = os.path.splitext(file_path)[0] + "_cropped_green.tif"
new_path_blue = os.path.splitext(file_path)[0] + "_cropped_blue.tif"
tf.imsave(new_path_red, cropped_red)
tf.imsave(new_path_green, cropped_green)
tf.imsave(new_path_blue, cropped_blue)
print(f"Cropped image saved to: {os.path.splitext(file_path)[0]}")

# show subplots per channel
fig2, axs2 = plt.subplots(1, 3, figsize=(15, 5))
axs2[0].imshow(cropped_red, cmap='Reds')
axs2[0].set_title("Red Channel")
axs2[0].axis('off')
axs2[1].imshow(cropped_green, cmap='Greens')
axs2[1].set_title("Green Channel")
axs2[1].axis('off')
axs2[2].imshow(cropped_blue, cmap='Blues')
axs2[2].set_title("Blue Channel")
axs2[2].axis('off')
plt.tight_layout()
plt.show()   
fig2.savefig(os.path.join(fig_save_path,'three_channels.eps'),
            format='eps',
            dpi=1000,           # controls raster‐embedded image resolution
            bbox_inches='tight')    

# check for pixels with perfect correlation
if cropped_red.shape != cropped_green.shape or cropped_red.shape != cropped_blue.shape:
    raise ValueError("Cropped images must have the same shape for all channels.")

# remove background - *2 allows for larger cells
background_subtract = True
if background_subtract:
    print("Subtracting background from red and green channels...")
    cropped_red_bs   = fj.subtract_background(cropped_red, tophat_radius=5, show=True, smoothing_sigma=1.5)
    cropped_green_bs = fj.subtract_background(cropped_green, tophat_radius=5, show=True, smoothing_sigma=1.5)
    cropped_blue_bs  = fj.subtract_background(cropped_blue, tophat_radius=5, show=True, smoothing_sigma=1.5)    
else:
    cropped_red_bs   = cropped_red.copy()
    cropped_green_bs = cropped_green.copy()
    cropped_blue_bs  = cropped_blue.copy()

# return cell diameter
#cell_diam = calibrate_cell_size(cropped_green_bs)
cell_diam = 5

# save out the background subtracted images
print("Saving background-subtracted images...")
new_path_red_bs = os.path.splitext(file_path)[0] + "_cropped_red_bs.tif"
new_path_green_bs = os.path.splitext(file_path)[0] + "_cropped_green_bs.tif"
new_path_blue_bs = os.path.splitext(file_path)[0] + "_cropped_blue_bs.tif"    
tf.imsave(new_path_red_bs, cropped_red_bs)
tf.imsave(new_path_green_bs, cropped_green_bs)
tf.imsave(new_path_blue_bs, cropped_blue_bs)    

# blob analysis

# Assume cropped_red, cropped_green, cropped_blue already defined
#rois_blue  = auto_detect_circular_rois(cropped_blue, cell_diameter=cell_diam, threshold_factor=0, smoothing_sigma=2.0, min_fraction=0.1)
#rois_red   = auto_detect_circular_rois(cropped_red_bs, cell_diameter=cell_diam, threshold_factor=0, smoothing_sigma=2.0, min_fraction=0.1)
#rois_green = auto_detect_circular_rois(cropped_green_bs, cell_diameter=cell_diam, threshold_factor=0, smoothing_sigma=2.0, min_fraction=0.1)

# filter out ROIs with too much overlap
#rois_red = filter_overlapping_rois(rois_red, cropped_red_bs, overlap_thresh=0.1)
#rois_green = filter_overlapping_rois(rois_green, cropped_green_bs, overlap_thresh=0.1)

# 
from scipy.stats import zscore
auto_thresh = ((np.ceil(cell_diam)/2)**2)*3.14
blobs_red   = fj.simple_blob_analysis(cropped_red_bs, min_sigma=2, max_sigma=9, num_sigma=5, threshold=10, threshold_factor=1, cmap='Reds')
blobs_green = fj.simple_blob_analysis(cropped_green_bs, min_sigma=2, max_sigma=9, num_sigma=5, threshold=10, threshold_factor=1, cmap='Greens')

# shared ROIs
shared_red_idx, shared_green_idx = fj.find_shared_rois(blobs_red, blobs_green)

fj.plot_shared_rois(cropped_red_bs, cropped_green_bs, cropped_blue_bs,
                blobs_red, blobs_green,
                shared_red_idx, shared_green_idx)

# store and save data
blob_data = dict()
blob_data['blobs_red'] = blobs_red
blob_data['blobs_green'] = blobs_green  
blob_data['shared_red_idx'] = shared_red_idx
blob_data['shared_green_idx'] = shared_green_idx
blob_data['cell_diameter'] = cell_diam  

# save the blob data to a .mat file
import scipy.io as sio
mat_save_path = os.path.splitext(file_path)[0] + "_blobs.mat"
sio.savemat(mat_save_path, blob_data)

# now identify percentage shared
percentage_red_in_green = len(shared_red_idx) / len(blobs_red) * 100 if blobs_red else 0
percentage_green_in_red = len(shared_green_idx) / len(blobs_green) * 100 if blobs_green else 0
print(f"Percentage of red ROIs in green: {percentage_red_in_green:.1f}%")
print(f"Percentage of green ROIs in red: {percentage_green_in_red:.1f}%")

# make a bar plot of the percentages
plt.figure(figsize=(8, 4))
plt.bar(['Red in Green', 'Green in Red'], 
        [percentage_red_in_green, percentage_green_in_red], 
        color=['red', 'green'])
plt.box(False)
plt.ylabel('Percentage of ROIs')
plt.title('Percentage of Shared ROIs')

# 3) visualize on the red and green channels as subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Red channel with ROIs (puncta analyzer)
axs[0].imshow(cropped_red_bs, cmap='Reds')
for cy, cx, radius in blobs_red:
    circ = Circle(
        (cx, cy),                # center
        radius,                  # radius
        edgecolor='black', 
        facecolor='none',
        linewidth=2
    )
    axs[0].add_patch(circ)
axs[0].set_title("Auto-detected circular ROIs (Red Channel)")

# Green channel with ROIs
axs[1].imshow(cropped_green_bs, cmap='Greens')
for cy, cx, radius in blobs_green:
    circ = Circle(
        (cx, cy),                # center
        radius,                  # radius
        edgecolor='black', 
        facecolor='none',
        linewidth=2
    )
    axs[1].add_patch(circ)
axs[1].set_title("Auto-detected circular ROIs (Green Channel)")
plt.tight_layout()
plt.show()

# build an svm to classify red_blobs and green_blobs using coordinates
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE

# make dataframe of red and green blobs
import pandas as pd
def make_blob_dataframe(blobs, label):
    data = []
    for cy, cx, radius in blobs:
        data.append({'y': cy, 'x': cx, 'radius': radius, 'label': label})
    return pd.DataFrame(data)

# make blob dataframes
df_red   = make_blob_dataframe(blobs_red, 'red')

# concatenate red and green blobs into a single dataframe
df_green = make_blob_dataframe(blobs_green, 'green')
df = pd.concat([df_red, df_green], ignore_index=True)

# drop the radius column
df = df.drop(columns=['radius'])

# split into features and labels
X = df[['y', 'x']].values  # features: y and x
y = df['label'].values      # labels: 'red' or 'green'

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# svm classifier
svm = SVC(kernel='poly', degree=5, C=1.0, random_state=42)

# fit the model
svm.fit(X_train, y_train)

# make predictions
y_pred = svm.predict(X_test)

# evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"SVM Accuracy: {accuracy:.2f}")

# show the classification line on the coordinates in X
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train == 'red', 1], X_train[y_train == 'red', 0], color='red', label='Red Blobs', alpha=0.5)
plt.scatter(X_train[y_train == 'green', 1], X_train[y_train == 'green', 0], color='green', label='Green Blobs', alpha=0.5)
plt.scatter(X_test[y_pred == 'red', 1], X_test[y_pred == 'red', 0], color='darkred', label='Predicted Red Blobs', marker='x')
plt.scatter(X_test[y_pred == 'green', 1], X_test[y_pred == 'green', 0], color='darkgreen', label='Predicted Green Blobs', marker='x')
plt.gca().invert_yaxis()
plt.title("SVM Classification of Red and Green Blobs")
plt.xlabel("X Coordinate")


# now plot the decision boundary
x_min, x_max = X[:, 1].min() - 1, X[:, 1].max() + 1
y_min, y_max = X[:, 0].min() - 1, X[:, 0].max() + 1
#xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
#                     np.arange(y_min, y_max, 0.1))  
    
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 10),  # 100 points per axis
    np.linspace(y_min, y_max, 10)
)    
Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, levels=[-1, 0, 1], alpha=0.2, cmap='coolwarm')
plt.legend()
plt.colorbar(label='Decision Function Value')
plt.show()



Beads 3